# Naive Bayes

Imports and helpers

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import joblib

## Data Acquisition

load data

In [ ]:
# Setup imports and helper functions
import sys
from pathlib import Path
root = Path('..')
sys.path.insert(0, str(root))
from src.utils import helpers
import os
import pandas as pd

# Ensure output directories
os.makedirs('../data/balanced100k', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../data/models', exist_ok=True)
os.makedirs('../data/predictions', exist_ok=True)

# If balanced dataset doesn't exist, create it from raw
balanced_dir = Path('../data/balanced100k')
balanced_dir.mkdir(parents=True, exist_ok=True)
balanced_name = 'balanced.csv'
balanced_path = balanced_dir / balanced_name
if not balanced_path.exists():
    print('Balanced dataset not found — creating from raw...')
    helpers.prepare_balanced_dataset(raw_path=None, balanced_dir=str(balanced_dir), balanced_name=balanced_name)
else:
    print('Balanced dataset found at', balanced_path)

# Load balanced dataset
balanced = helpers.load_balanced(str(balanced_path))
print('Loaded balanced rows:', len(balanced))


display data

## Exploratory Data Analysis - EDA

Check Missing Values

In [ ]:
# EDA on balanced dataset
print('Columns:', list(balanced.columns))
print('\nClass distribution:')
print(balanced['__y__'].value_counts())

text_candidates = [c for c in balanced.columns if balanced[c].dtype == 'object']
text_col = text_candidates[0] if text_candidates else None
balanced['text_len'] = balanced[text_col].astype(str).str.len()
print('\nText length describe:')
print(balanced['text_len'].describe())
print('\nSample texts:')
print(balanced[text_col].dropna().sample(n=5, random_state=42).tolist())


Check Class Balance

Text Analysis

## Data Preprocessing

## Feature Engineering

## Model Training

## Model Evaluation

## Model Testing

## Saving the Model and Pipeline

## Insights

In [ ]:
# Load balanced dataset created by notebook 01
balanced_path = '../data/balanced100k/balanced.csv'
print('Loading balanced dataset from', balanced_path)
balanced = pd.read_csv(balanced_path)
print('Rows:', len(balanced))


In [ ]:
# Detect text and target (aligns with notebook 01)
text_candidates = [c for c in balanced.columns if balanced[c].dtype == 'object']
text_col = None
for c in text_candidates:
    if balanced[c].astype(str).str.len().median() > 20:
        text_col = c
        break
if text_col is None and text_candidates:
    text_col = text_candidates[0]
if '__y__' in balanced.columns:
    y = balanced['__y__']
else:
    # fallback: try common target names
    for candidate in ['label','sentiment','rating','score','stars','target']:
        if candidate in balanced.columns:
            y = balanced[candidate]
            break
print('Using text column:', text_col)


In [ ]:
# Train/test split and vectorization
X = balanced[text_col].fillna('')
y = balanced['__y__'] if '__y__' in balanced.columns else balanced[y.name]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
vectorizer = TfidfVectorizer(max_features=50000, ngram_range=(1,2))
X_train_t = vectorizer.fit_transform(X_train)
X_test_t = vectorizer.transform(X_test)

# Train Multinomial Naive Bayes
clf = MultinomialNB(alpha=1.0)
clf.fit(X_train_t, y_train)
y_pred = clf.predict(X_test_t)
print('Accuracy:', accuracy_score(y_test, y_pred))
print('\nClassification report:\n', classification_report(y_test, y_pred))

# Save model and predictions
joblib.dump(clf, 'data/models/naive_bayes.joblib')
joblib.dump(vectorizer, 'data/models/tfidf_vectorizer_nb.joblib')
preds_df = X_test.to_frame(name=text_col) if isinstance(X_test, pd.Series) else pd.DataFrame({text_col: X_test})
preds_df['y_true'] = y_test.values
preds_df['y_pred'] = y_pred
preds_df.to_csv('data/predictions/naive_bayes_predictions.csv', index=False)
print('Saved Naive Bayes model and predictions')
